# 🧬 Pediatric Genetic Disorder — Target 1: Genetic Disorder Type
## Model Training Pipeline (4-Class Classification)

> **Goal:** Predict `Genetic Disorder` type (4 classes: 0, 1, 2, 3)

### Strategy
1. Load preprocessed data
2. Train individual models (XGBoost, LightGBM, CatBoost, RandomForest)
3. Optuna-based hyperparameter tuning for top models
4. Stacking Ensemble for best performance
5. Generate predictions on test set


In [ ]:
# ── Install / imports ───────────────────────────────────────────────────────────
!pip install -q optuna lightgbm catboost xgboost

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import time
warnings.filterwarnings('ignore')

from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    StackingClassifier, VotingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, roc_auc_score
)
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

np.random.seed(42)
pd.set_option('display.max_columns', None)
print('✅ Libraries loaded')

In [ ]:
# ── Load preprocessed data ──────────────────────────────────────────────────────
# Update paths to match your Kaggle dataset input folder
DATA_PATH = '/kaggle/input/datasets/masteranany/newgec-preprocessed/'

X_train = pd.read_csv(DATA_PATH + 'X_train_preprocessed.csv')
X_test  = pd.read_csv(DATA_PATH + 'X_test_preprocessed.csv')
y_train = np.load(DATA_PATH + 'y1_train.npy')   # Target 1: Genetic Disorder

print(f'X_train shape : {X_train.shape}')
print(f'X_test  shape : {X_test.shape}')
print(f'y_train shape : {y_train.shape}')
print(f'Classes       : {np.unique(y_train)}')
print(f'Class counts  : {np.bincount(y_train)}')
print(f'Null in X_train: {X_train.isnull().sum().sum()}')
print(f'Null in X_test : {X_test.isnull().sum().sum()}')

In [ ]:
# ── Target class distribution ───────────────────────────────────────────────────
class_names = {
    0: 'Mitochondrial',
    1: 'Multifactorial',
    2: 'Single-gene',
    3: 'Unknown'
}

fig, ax = plt.subplots(figsize=(8, 4))
counts = np.bincount(y_train)
bars = ax.bar([class_names.get(i, str(i)) for i in range(len(counts))],
              counts, color=sns.color_palette('tab10', len(counts)))
ax.set_title('Target 1 — Genetic Disorder Class Distribution (After SMOTE)', fontsize=13)
ax.set_ylabel('Count')
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            str(count), ha='center', fontsize=10)
plt.tight_layout(); plt.show()
print(f'Total training samples: {len(y_train)}')

In [ ]:
# ── Cross-validation setup ──────────────────────────────────────────────────────
N_SPLITS = 5
N_CLASSES = len(np.unique(y_train))
CV = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

def evaluate_model(model, X, y, cv, model_name='Model'):
    """Evaluate model via cross-validation and return metrics."""
    start = time.time()
    scores = cross_val_score(model, X, y,
                             cv=cv,
                             scoring='f1_macro',
                             n_jobs=-1)
    elapsed = time.time() - start
    print(f'{model_name:40s} | F1-macro: {scores.mean():.4f} ± {scores.std():.4f} | Time: {elapsed:.1f}s')
    return scores.mean(), scores.std()

results = {}
print(f'\n{"Model":40s} | {"F1-macro (CV5)":20s} | Time')
print('-' * 80)

In [ ]:
# ── Baseline Models ─────────────────────────────────────────────────────────────
print('\n=== BASELINE MODELS ===\n')

# 1. Random Forest
rf = RandomForestClassifier(n_estimators=300, max_depth=None, min_samples_split=2,
                             n_jobs=-1, random_state=42, class_weight='balanced')
mean, std = evaluate_model(rf, X_train, y_train, CV, 'RandomForest (baseline)')
results['RF_baseline'] = mean

# 2. XGBoost
xgb = XGBClassifier(n_estimators=300, learning_rate=0.1, max_depth=6,
                     subsample=0.8, colsample_bytree=0.8,
                     eval_metric='mlogloss',
                     random_state=42, n_jobs=-1)
mean, std = evaluate_model(xgb, X_train, y_train, CV, 'XGBoost (baseline)')
results['XGB_baseline'] = mean

# 3. LightGBM
lgb = LGBMClassifier(n_estimators=300, learning_rate=0.1, max_depth=-1,
                     num_leaves=31, subsample=0.8, colsample_bytree=0.8,
                     random_state=42, n_jobs=-1, verbose=-1)
mean, std = evaluate_model(lgb, X_train, y_train, CV, 'LightGBM (baseline)')
results['LGB_baseline'] = mean

# 4. CatBoost
cat = CatBoostClassifier(iterations=300, learning_rate=0.1, depth=6,
                          random_seed=42, verbose=0)
mean, std = evaluate_model(cat, X_train, y_train, CV, 'CatBoost (baseline)')
results['CAT_baseline'] = mean

print('\nBaseline results summary:', {k: f'{v:.4f}' for k, v in results.items()})

In [ ]:
# ── Optuna Tuning: XGBoost ──────────────────────────────────────────────────────
print('\n=== OPTUNA TUNING: XGBoost ===')

def xgb_objective(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 200, 800),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth':         trial.suggest_int('max_depth', 3, 10),
        'subsample':         trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.6, 1.0),
        'min_child_weight':  trial.suggest_int('min_child_weight', 1, 10),
        'gamma':             trial.suggest_float('gamma', 0, 5),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-5, 10, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-5, 10, log=True),
        'use_label_encoder': False,
        'eval_metric':       'mlogloss',
        'random_state':      42,
        'n_jobs':            -1
    }
    model = XGBClassifier(**params)
    scores = cross_val_score(model, X_train, y_train, cv=CV,
                             scoring='f1_macro', n_jobs=-1)
    return scores.mean()

xgb_study = optuna.create_study(direction='maximize',
                                 sampler=optuna.samplers.TPESampler(seed=42))
xgb_study.optimize(xgb_objective, n_trials=50, show_progress_bar=True)

print(f'\nBest XGB F1-macro: {xgb_study.best_value:.4f}')
print(f'Best XGB params  : {xgb_study.best_params}')

In [ ]:
# ── Optuna Tuning: LightGBM ─────────────────────────────────────────────────────
print('\n=== OPTUNA TUNING: LightGBM ===')

def lgb_objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 200, 1000),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves':       trial.suggest_int('num_leaves', 20, 200),
        'max_depth':        trial.suggest_int('max_depth', 3, 12),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_samples':trial.suggest_int('min_child_samples', 5, 50),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-5, 10, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-5, 10, log=True),
        'random_state':     42,
        'n_jobs':           -1,
        'verbose':          -1
    }
    model = LGBMClassifier(**params)
    scores = cross_val_score(model, X_train, y_train, cv=CV,
                             scoring='f1_macro', n_jobs=-1)
    return scores.mean()

lgb_study = optuna.create_study(direction='maximize',
                                  sampler=optuna.samplers.TPESampler(seed=42))
lgb_study.optimize(lgb_objective, n_trials=50, show_progress_bar=True)

print(f'\nBest LGB F1-macro: {lgb_study.best_value:.4f}')
print(f'Best LGB params  : {lgb_study.best_params}')

In [ ]:
# ── Optuna Tuning: CatBoost ─────────────────────────────────────────────────────
print('\n=== OPTUNA TUNING: CatBoost ===')

def cat_objective(trial):
    params = {
        'iterations':     trial.suggest_int('iterations', 200, 800),
        'learning_rate':  trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'depth':          trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg':    trial.suggest_float('l2_leaf_reg', 1e-5, 10, log=True),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0, 1),
        'random_strength':trial.suggest_float('random_strength', 0, 1),
        'border_count':   trial.suggest_int('border_count', 32, 255),
        'random_seed':    42,
        'verbose':        0
    }
    model = CatBoostClassifier(**params)
    scores = cross_val_score(model, X_train, y_train, cv=CV,
                             scoring='f1_macro', n_jobs=1)
    return scores.mean()

cat_study = optuna.create_study(direction='maximize',
                                  sampler=optuna.samplers.TPESampler(seed=42))
cat_study.optimize(cat_objective, n_trials=40, show_progress_bar=True)

print(f'\nBest CAT F1-macro: {cat_study.best_value:.4f}')
print(f'Best CAT params  : {cat_study.best_params}')

In [ ]:
# ── Optuna Tuning: Random Forest ────────────────────────────────────────────────
print('\n=== OPTUNA TUNING: Random Forest ===')

def rf_objective(trial):
    params = {
        'n_estimators':   trial.suggest_int('n_estimators', 200, 600),
        'max_depth':      trial.suggest_int('max_depth', 5, 30),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'min_samples_leaf':  trial.suggest_int('min_samples_leaf', 1, 5),
        'max_features':   trial.suggest_categorical('max_features', ['sqrt', 'log2', 0.5, 0.7]),
        'class_weight':   'balanced',
        'n_jobs':         -1,
        'random_state':   42
    }
    model = RandomForestClassifier(**params)
    scores = cross_val_score(model, X_train, y_train, cv=CV,
                             scoring='f1_macro', n_jobs=-1)
    return scores.mean()

rf_study = optuna.create_study(direction='maximize',
                                 sampler=optuna.samplers.TPESampler(seed=42))
rf_study.optimize(rf_objective, n_trials=30, show_progress_bar=True)

print(f'\nBest RF  F1-macro: {rf_study.best_value:.4f}')
print(f'Best RF  params  : {rf_study.best_params}')

In [ ]:
# ── Tuning Summary ───────────────────────────────────────────────────────────────
print('\n=== TUNING SUMMARY ===')
tuning_results = {
    'XGBoost (tuned)':    xgb_study.best_value,
    'LightGBM (tuned)':   lgb_study.best_value,
    'CatBoost (tuned)':   cat_study.best_value,
    'RandomForest (tuned)': rf_study.best_value,
}
for name, score in sorted(tuning_results.items(), key=lambda x: -x[1]):
    print(f'  {name:30s}: {score:.4f}')

# Visualization
fig, ax = plt.subplots(figsize=(8, 4))
names = list(tuning_results.keys())
scores = list(tuning_results.values())
colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']
bars = ax.bar(names, scores, color=colors, alpha=0.85)
ax.set_ylim(min(scores) - 0.02, 1.0)
ax.set_ylabel('F1-macro (CV5)')
ax.set_title('Target 1 — Model Comparison After Hyperparameter Tuning')
for bar, score in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{score:.4f}', ha='center', fontsize=10, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── Build Tuned Models ──────────────────────────────────────────────────────────
print('\n=== BUILDING TUNED MODELS ===')

xgb_best = XGBClassifier(
    **xgb_study.best_params,
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=42, n_jobs=-1
)

lgb_best = LGBMClassifier(
    **lgb_study.best_params,
    random_state=42, n_jobs=-1, verbose=-1
)

cat_best = CatBoostClassifier(
    **cat_study.best_params,
    random_seed=42, verbose=0
)

rf_best = RandomForestClassifier(
    **rf_study.best_params,
    class_weight='balanced',
    n_jobs=-1, random_state=42
)

print('Tuned models created ✅')

In [ ]:
# ── Stacking Ensemble ───────────────────────────────────────────────────────────
print('\n=== STACKING ENSEMBLE ===')

# Base estimators: top performers
base_estimators = [
    ('xgb', xgb_best),
    ('lgb', lgb_best),
    ('cat', cat_best),
    ('rf',  rf_best),
]

# Meta-learner: Logistic Regression (robust for stacking)
meta_learner = LogisticRegression(
    max_iter=1000,
    C=1.0,
    solver='lbfgs',
    multi_class='auto',
    random_state=42
)

stack = StackingClassifier(
    estimators=base_estimators,
    final_estimator=meta_learner,
    cv=5,                 # inner CV for generating meta-features
    passthrough=False,    # only pass predictions, not raw features
    n_jobs=-1
)

# Evaluate stacking via outer CV
print('Evaluating stacking ensemble (this may take a few minutes)...')
stack_scores = cross_val_score(
    stack, X_train, y_train,
    cv=CV, scoring='f1_macro', n_jobs=1
)
print(f'\nStacking F1-macro: {stack_scores.mean():.4f} ± {stack_scores.std():.4f}')

In [ ]:
# ── Soft-Voting Ensemble ────────────────────────────────────────────────────────
print('\n=== SOFT-VOTING ENSEMBLE ===')

voting = VotingClassifier(
    estimators=base_estimators,
    voting='soft',
    n_jobs=-1
)

voting_scores = cross_val_score(
    voting, X_train, y_train,
    cv=CV, scoring='f1_macro', n_jobs=1
)
print(f'Voting  F1-macro: {voting_scores.mean():.4f} ± {voting_scores.std():.4f}')

In [ ]:
# ── Final Comparison ─────────────────────────────────────────────────────────────
print('\n=== FINAL MODEL COMPARISON ===')
all_results = {
    **tuning_results,
    'Stacking Ensemble': stack_scores.mean(),
    'Voting  Ensemble':  voting_scores.mean(),
}
for name, score in sorted(all_results.items(), key=lambda x: -x[1]):
    print(f'  {name:35s}: {score:.4f}')

best_model_name = max(all_results, key=all_results.get)
print(f'\n🏆 Best model: {best_model_name} — F1: {all_results[best_model_name]:.4f}')

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
names  = list(all_results.keys())
scores = list(all_results.values())
colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#E91E63', '#00BCD4']
bars = ax.bar(names, scores, color=colors[:len(names)], alpha=0.85)
ax.set_ylim(min(scores) - 0.02, 1.0)
ax.set_ylabel('F1-macro (CV5)')
ax.set_title('Target 1 — All Models Comparison')
plt.xticks(rotation=30, ha='right')
for bar, score in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{score:.4f}', ha='center', fontsize=9, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── Train Best Model on Full Training Data ──────────────────────────────────────
print('\n=== TRAINING BEST MODEL ON FULL DATA ===')

# Pick the best model object (Stacking is usually best)
# Adjust if Voting or a single model wins
best_model_map = {
    'XGBoost (tuned)':       xgb_best,
    'LightGBM (tuned)':      lgb_best,
    'CatBoost (tuned)':      cat_best,
    'RandomForest (tuned)':  rf_best,
    'Stacking Ensemble':     stack,
    'Voting  Ensemble':      voting,
}

final_model = best_model_map.get(best_model_name, stack)

print(f'Fitting: {best_model_name} ...')
final_model.fit(X_train, y_train)
print('Fitting complete ✅')

# Training set evaluation (sanity check)
y_pred_train = final_model.predict(X_train)
print(f'\nTraining accuracy : {accuracy_score(y_train, y_pred_train):.4f}')
print(f'Training F1-macro : {f1_score(y_train, y_pred_train, average="macro"):.4f}')

In [ ]:
# ── Full CV Report on Best Model ─────────────────────────────────────────────────
print('\n=== DETAILED CV REPORT (Best Model) ===')

cv_preds = np.zeros(len(y_train), dtype=int)
for fold, (tr_idx, val_idx) in enumerate(CV.split(X_train, y_train)):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train[tr_idx], y_train[val_idx]

    fold_model = best_model_map.get(best_model_name, stack)
    fold_model.fit(X_tr, y_tr)
    cv_preds[val_idx] = fold_model.predict(X_val)

    fold_f1 = f1_score(y_val, cv_preds[val_idx], average='macro')
    print(f'  Fold {fold+1}: F1-macro = {fold_f1:.4f}')

overall_f1 = f1_score(y_train, cv_preds, average='macro')
overall_acc = accuracy_score(y_train, cv_preds)
print(f'\nOOF F1-macro : {overall_f1:.4f}')
print(f'OOF Accuracy : {overall_acc:.4f}')
print(f'\n{classification_report(y_train, cv_preds)}')

In [ ]:
# ── Confusion Matrix ─────────────────────────────────────────────────────────────
cm = confusion_matrix(y_train, cv_preds)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=[class_names.get(i, str(i)) for i in range(N_CLASSES)],
            yticklabels=[class_names.get(i, str(i)) for i in range(N_CLASSES)])
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Target 1 — OOF Confusion Matrix')
plt.tight_layout(); plt.show()

In [ ]:
# ── Feature Importance ──────────────────────────────────────────────────────────
print('\n=== FEATURE IMPORTANCE ===')

# Use LightGBM or XGBoost for importance (single-model)
fi_model = lgb_best
fi_model.fit(X_train, y_train)

importances = pd.Series(fi_model.feature_importances_,
                         index=X_train.columns).sort_values(ascending=False)

top_n = 20
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(x=importances.values[:top_n],
            y=importances.index[:top_n],
            ax=ax, palette='viridis')
ax.set_title(f'Top {top_n} Feature Importances (LightGBM)')
ax.set_xlabel('Importance')
plt.tight_layout(); plt.show()

print('\nTop 10 features:')
print(importances.head(10).to_string())

In [ ]:
# ── Generate Test Predictions ───────────────────────────────────────────────────
print('\n=== GENERATING TEST PREDICTIONS ===')

# Re-fit final model on full train data (in case CV loop changed it)
final_model.fit(X_train, y_train)

y_test_pred     = final_model.predict(X_test)
y_test_pred_proba = final_model.predict_proba(X_test)

print(f'Test predictions shape      : {y_test_pred.shape}')
print(f'Test predictions distribution: {np.bincount(y_test_pred)}')
print(f'Unique classes predicted    : {np.unique(y_test_pred)}')

In [ ]:
# ── Save Predictions ─────────────────────────────────────────────────────────────
import pickle, os

output_dir = '/kaggle/working/'
os.makedirs(output_dir, exist_ok=True)

# Save hard predictions
np.save(output_dir + 'y1_test_pred.npy', y_test_pred)

# Save probability predictions
np.save(output_dir + 'y1_test_pred_proba.npy', y_test_pred_proba)

# Save as CSV for submission
submission = pd.DataFrame({
    'index': range(len(y_test_pred)),
    'Genetic_Disorder': y_test_pred
})
submission.to_csv(output_dir + 'submission_target1.csv', index=False)

# Save model
with open(output_dir + 'model_target1.pkl', 'wb') as f:
    pickle.dump(final_model, f)

print(f'✅ Predictions saved:')
print(f'   {output_dir}y1_test_pred.npy')
print(f'   {output_dir}y1_test_pred_proba.npy')
print(f'   {output_dir}submission_target1.csv')
print(f'   {output_dir}model_target1.pkl')

print(f'\n=== FINAL SUMMARY ===')
print(f'Best model       : {best_model_name}')
print(f'Best CV F1-macro : {all_results[best_model_name]:.4f}')
print(f'OOF F1-macro     : {overall_f1:.4f}')
print(f'OOF Accuracy     : {overall_acc:.4f}')